In [0]:
%pip install -U mlflow
dbutils.library.restartPython()
%pip install openai


In [0]:
import mlflow
mlflow.openai.autolog()

In [0]:
# Databricks Notebook or Workflow Task
from pyspark.sql import functions as F
import openai

# Get your OpenAI or Mosaic API key securely
client = openai.OpenAI(api_key=dbutils.secrets.get("openai", "api_key"))

# Step 1. Fetch latest failed Databricks job from system tables
df_failed = (
    spark.read.table("system.lakeflow.job_run_timeline")
    .filter(F.col("result_state") == "ERROR")
    .orderBy(F.col("period_start_time").desc())
    .limit(1)
)

failed_job = df_failed.collect()[0]

job_id = failed_job["job_id"]
run_id = failed_job["run_id"]
notebook_path = failed_job.get("task", {}).get("notebook_task", {}).get("notebook_path", "N/A")
error_message = failed_job["error_message"]
stack_trace = failed_job.get("stack_trace", "No stack trace available")

# Step 2. Generate a prompt for the LLM
prompt = f"""
A Databricks job failed.

Job ID: {job_id}
Run ID: {run_id}
Notebook Path: {notebook_path}
Error Message: {error_message}
Stack Trace: {stack_trace}

Analyze this failure and provide:
1. The most likely root cause.
2. Recommended fix or preventive action.
"""

# Step 3. Ask LLM (OpenAI or Mosaic AI) for root cause analysis
response = client.chat.completions.create(
    model="gpt-4-turbo",  # or your MosaicAI model endpoint
    messages=[
        {"role": "system", "content": "You are a Databricks job failure analysis expert."},
        {"role": "user", "content": prompt},
    ],
)

# Step 4. Print AI-generated root cause summary
print("🔍 Root Cause Analysis:")
print(response.choices[0].message.content)

In [0]:
%sql
select * from system.lakeflow.job_run_timeline
limit 100